# SceneScope Mood Classifier Fine-Tuning (Colab)

This notebook applies the production fixes:
- **No `min_count` downsampling** (keep much more data)
- **Class-weighted loss** instead of hard balancing
- **Dynamic padding** (memory efficient)
- **T4-safe defaults** with gradient accumulation
- Optional HF Hub push at the end


In [ ]:
!pip -q install -U transformers datasets evaluate scikit-learn accelerate pandas numpy huggingface_hub

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import torch

from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, f1_score, classification_report

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Paths (change if needed)
BASE_DIR = "/content"
RAW_CSV = os.path.join(BASE_DIR, "screenplay_moods_v3_raw.csv")

# Labels used by SceneScope backend
VALID_LABELS = ["tense", "somber", "uplifting", "romantic", "comedic"]

# Model
MODEL_NAME = "roberta-base"

# Data filtering knobs
MIN_CONFIDENCE = 0.60   # was often 0.70; lower keeps more usable training rows
MAX_TEXT_LEN_CHARS = 2000

# Tokenization/training knobs
MAX_LENGTH = 256
NUM_EPOCHS = 4
TRAIN_BATCH = 8          # T4-safe
EVAL_BATCH = 8
GRAD_ACCUM = 2           # effective train batch = 16
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
OUTPUT_DIR = "/content/scenescope_roberta_weighted"


In [ ]:
# Load raw labeled data (from your labeling pipeline)
assert os.path.exists(RAW_CSV), f"Missing file: {RAW_CSV}"
df = pd.read_csv(RAW_CSV)

# Expected columns from your existing notebook: scene_text, label, confidence
required = {"scene_text", "label"}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {missing}")

if "confidence" in df.columns:
    df = df[df["confidence"].fillna(0.0) >= MIN_CONFIDENCE].copy()

df = df[df["label"].isin(VALID_LABELS)].copy()
df["scene_text"] = df["scene_text"].astype(str).str.slice(0, MAX_TEXT_LEN_CHARS)
df = df.dropna(subset=["scene_text", "label"])
df = df[df["scene_text"].str.len() > 20].reset_index(drop=True)

print("Rows kept:", len(df))
print(df["label"].value_counts())


In [ ]:
# Label mapping
labels = sorted(df["label"].unique())
label2id = {l: i for i, l in enumerate(labels)}
id2label = {i: l for l, i in label2id.items()}
df["label_id"] = df["label"].map(label2id)

# Split
train_df, temp_df = train_test_split(
    df,
    test_size=0.2,
    random_state=SEED,
    stratify=df["label"],
)
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=SEED,
    stratify=temp_df["label"],
)

print("Train/Val/Test:", len(train_df), len(val_df), len(test_df))
print("Train label dist:\n", train_df["label"].value_counts())

# Class weights for weighted CE loss
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array(sorted(train_df["label_id"].unique())),
    y=train_df["label_id"].values,
)
class_weights = torch.tensor(class_weights, dtype=torch.float)
print("Class weights:", class_weights)


In [ ]:
dataset = DatasetDict({
    "train": Dataset.from_pandas(train_df[["scene_text", "label_id"]].rename(columns={"label_id": "label"})),
    "validation": Dataset.from_pandas(val_df[["scene_text", "label_id"]].rename(columns={"label_id": "label"})),
    "test": Dataset.from_pandas(test_df[["scene_text", "label_id"]].rename(columns={"label_id": "label"})),
})

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(batch["scene_text"], truncation=True, max_length=MAX_LENGTH)

tokenized = dataset.map(tokenize, batched=True)
tokenized = tokenized.remove_columns(["scene_text", "__index_level_0__"])

# Dynamic padding -> better memory use than padding='max_length' everywhere
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)


In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id,
)

def compute_metrics(eval_pred):
    logits, y_true = eval_pred
    y_pred = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "f1_macro": f1_score(y_true, y_pred, average="macro"),
        "f1_weighted": f1_score(y_true, y_pred, average="weighted"),
    }

class WeightedTrainer(Trainer):
    def __init__(self, class_weights=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        cw = self.class_weights.to(logits.device) if self.class_weights is not None else None
        loss_fct = torch.nn.CrossEntropyLoss(weight=cw)
        loss = loss_fct(logits.view(-1, model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss


In [ ]:
fp16 = torch.cuda.is_available()
bf16 = False  # set True only on supported GPUs (A100/H100 class)

args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=50,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=TRAIN_BATCH,
    per_device_eval_batch_size=EVAL_BATCH,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=NUM_EPOCHS,
    weight_decay=WEIGHT_DECAY,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    fp16=fp16,
    bf16=bf16,
    report_to="none",
    seed=SEED,
)

trainer = WeightedTrainer(
    model=model,
    args=args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    class_weights=class_weights,
)

trainer.train()


In [ ]:
metrics = trainer.evaluate(tokenized["test"])
print("Test metrics:", metrics)

preds_output = trainer.predict(tokenized["test"])
y_true_ids = preds_output.label_ids
y_pred_ids = np.argmax(preds_output.predictions, axis=-1)

y_true = [id2label[i] for i in y_true_ids]
y_pred = [id2label[i] for i in y_pred_ids]
print(classification_report(y_true, y_pred, labels=labels, digits=3))

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("Saved to", OUTPUT_DIR)


## Optional: push to Hugging Face Hub

```python
from huggingface_hub import login
login()  # paste token

REPO_NAME = "RedMinder56/scenescope-mood-classifier"
trainer.push_to_hub(REPO_NAME)
tokenizer.push_to_hub(REPO_NAME)
```
